# 05 - Hidden-feature steering and method comparison

Notebook `04` obtained a steering direction by differentiating an external
objective at every active step. Here we ask whether a useful direction is
already present inside the frozen velocity network. We record one hidden layer
on labeled, noised examples, test on separate examples whether class can be
decoded from that representation, and learn a target-versus-rest direction
offline. During sampling we add that direction to the hidden feature rather
than to the sample position, so the downstream layers decide how the velocity
changes and no inference-time gradient is needed.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, _ = load_or_train_model(device=device)
reference_data, reference_labels = sample_labeled_mixture(30000, device=device)
stats = estimate_gaussian_stats(reference_data, reference_labels)
TARGET_CLASS = 2
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
checkpoint_digest_before = model_state_digest(model)


## 1. Why intervene inside the network?

The velocity MLP first converts the current state `(t, x_t)` into a hidden
feature vector $h_t$. The intervention changes that feature before the final
layers compute velocity:

$$h'_t = h_t + \alpha\,g(t)\,\operatorname{RMS}(h_t)\,d_{0.90}.$$

Here $d_{0.90}$ is a unit target-vs-rest direction learned from a labeled
activation training split and checked on separate held-out activations,
$\alpha$ controls strength, and $g(t)$ turns the edit on only in a chosen time
window. The RMS factor puts the unit direction on the scale of the current
layer.

This does **not** pull the sample toward a point. It changes an internal feature,
which may change the model's velocity; the ODE sampler then accumulates those
velocity changes. Model weights stay frozen.


In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

def activation_box(ax, xy, width, text, color):
    patch = FancyBboxPatch(
        xy, width, 0.18,
        boxstyle="round,pad=0.018,rounding_size=0.018",
        facecolor=color, edgecolor="#333333", linewidth=1.1,
    )
    ax.add_patch(patch)
    ax.text(xy[0] + width / 2, xy[1] + 0.09, text, ha="center", va="center", fontsize=9.5)

def activation_arrow(ax, start, end, color="#333333"):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=14, linewidth=1.35, color=color))

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis("off")

ax.text(0.01, 0.80, "Ordinary forward pass", fontsize=11, weight="bold")
activation_box(ax, (0.03, 0.56), 0.14, "state\n$(t,x_t)$", "#E7E6E6")
activation_box(ax, (0.25, 0.56), 0.16, "hidden feature\n$h_t$", "#CFE2F3")
activation_box(ax, (0.51, 0.56), 0.18, "later MLP\nlayers", "#D9EAD3")
activation_box(ax, (0.79, 0.56), 0.17, "velocity\n$v_\\theta(t,x_t)$", "#EAD1DC")
activation_arrow(ax, (0.17, 0.65), (0.25, 0.65))
activation_arrow(ax, (0.41, 0.65), (0.51, 0.65))
activation_arrow(ax, (0.69, 0.65), (0.79, 0.65))

ax.text(0.01, 0.34, "Steered forward pass", fontsize=11, weight="bold")
activation_box(ax, (0.03, 0.10), 0.14, "same state\n$(t,x_t)$", "#E7E6E6")
activation_box(ax, (0.25, 0.10), 0.16, "same hidden\n$h_t$", "#CFE2F3")
activation_box(ax, (0.48, 0.10), 0.24, "edited feature\n$h'_t=h_t+\\alpha g(t)\\,\\mathrm{RMS}(h_t)d_{0.90}$", "#FCE5CD")
activation_box(ax, (0.79, 0.10), 0.17, "changed\nvelocity", "#EAD1DC")
activation_arrow(ax, (0.17, 0.19), (0.25, 0.19))
activation_arrow(ax, (0.41, 0.19), (0.48, 0.19))
activation_arrow(ax, (0.72, 0.19), (0.79, 0.19))
ax.text(0.60, 0.41, "learned direction $d_{0.90}$", ha="center", fontsize=9.5, color="#D55E00")
activation_arrow(ax, (0.60, 0.39), (0.60, 0.29), color="#D55E00")

fig.suptitle("Activation steering edits a hidden feature, not the sample position", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()


## 2. When is class information visible in the hidden layer?

At each time, we record activations from separate balanced training and test
sets with independent noise. A linear probe is fitted only on the training
activations and scored on the untouched test activations. Shuffled-label probes
show how much apparent accuracy can arise without the true class relationship.
The comparison asks whether class is readable from the hidden state; it does
not yet show that editing the state will control generation.


In [ ]:
def wilson_interval(successes, total, z=1.96):
    proportion = successes / total
    denominator = 1 + z**2 / total
    center = (proportion + z**2 / (2 * total)) / denominator
    radius = z * np.sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
    return center - radius, center + radius

probe_rows = []
directions = {}
probes = {}
feature_sets = {}
for time_index, t_value in enumerate([0.10, 0.25, 0.40, 0.55, 0.70, 0.85, 0.90]):
    set_seed(SEED + 500 + 2 * time_index)
    train_features, train_labels = collect_forward_activations(
        model, t_value=t_value, n_per_class=600
    )
    set_seed(SEED + 501 + 2 * time_index)
    test_features, test_labels = collect_forward_activations(
        model, t_value=t_value, n_per_class=400
    )

    probe = fit_linear_probe(train_features, train_labels)
    train_accuracy = (probe.predict(train_features) == train_labels).float().mean().item()
    test_predictions = probe.predict(test_features)
    test_successes = int((test_predictions == test_labels).sum().item())
    test_accuracy = test_successes / test_labels.numel()
    interval_low, interval_high = wilson_interval(test_successes, test_labels.numel())

    shuffled_accuracies = []
    for null_index in range(20):
        set_seed(SEED + 2000 + 100 * time_index + null_index)
        shuffled_labels = train_labels[torch.randperm(train_labels.numel(), device=device)]
        shuffled_probe = fit_linear_probe(train_features, shuffled_labels)
        shuffled_accuracy = (
            shuffled_probe.predict(test_features) == test_labels
        ).float().mean().item()
        shuffled_accuracies.append(shuffled_accuracy)

    direction = discriminant_direction(train_features, train_labels, TARGET_CLASS)
    directions[t_value] = direction
    probes[t_value] = probe
    feature_sets[t_value] = (train_features, train_labels, test_features, test_labels)
    probe_rows.append({
        "t": t_value,
        "effective noise std": BASE_STD * (1-t_value) / t_value,
        "training accuracy": train_accuracy,
        "held-out accuracy": test_accuracy,
        "95% interval low": interval_low,
        "95% interval high": interval_high,
        "shuffled mean": float(np.mean(shuffled_accuracies)),
        "shuffled std": float(np.std(shuffled_accuracies)),
        "shuffled 95th percentile": float(np.quantile(shuffled_accuracies, 0.95)),
        "shuffled max": float(np.max(shuffled_accuracies)),
    })

probe_table = pd.DataFrame(probe_rows)
display(probe_table.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
lower_error = np.maximum(
    0.0, probe_table["held-out accuracy"] - probe_table["95% interval low"]
)
upper_error = np.maximum(
    0.0, probe_table["95% interval high"] - probe_table["held-out accuracy"]
)
ax.errorbar(
    probe_table["t"], probe_table["held-out accuracy"],
    yerr=np.vstack([lower_error, upper_error]), marker="o", capsize=3,
    color="#0072B2", label="held-out true-label probe",
)
ax.plot(probe_table["t"], probe_table["shuffled mean"], color="#D55E00", label="20 shuffled-label probes")
ax.fill_between(
    probe_table["t"],
    probe_table["shuffled mean"] - probe_table["shuffled std"],
    probe_table["shuffled mean"] + probe_table["shuffled std"],
    color="#D55E00", alpha=0.18,
)
ax.axhline(1 / len(CLASS_NAMES), color="#777777", ls="--", label="chance")
ax.set(xlabel="t (noise -> data)", ylabel="held-out accuracy", ylim=(0.25, 1.02), title="Class information across time")
ax.annotate("near chance", (0.10, probe_table.loc[0, "held-out accuracy"]), xytext=(8, 14), textcoords="offset points")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
plt.show()


In [ ]:
# Fit display PCA only on training activations, then transform held-out activations.
REFERENCE_T = 0.90
train_features, train_labels, test_features, test_labels = feature_sets[REFERENCE_T]
pca_display = fit_pca_projection(train_features)
projected_test = pca_display.transform(test_features)

fig, ax = plt.subplots(figsize=(6, 5))
plot_labeled_points(
    projected_test, test_labels, ax=ax,
    title=f"Held-out activations in train-fit PCA coordinates at t={REFERENCE_T}", alpha=0.35,
)
ax.legend(frameon=False)
plt.show()

reference_predictions = probes[REFERENCE_T].predict(test_features)
confusion = torch.bincount(
    test_labels * len(CLASS_NAMES) + reference_predictions,
    minlength=len(CLASS_NAMES) ** 2,
).reshape(len(CLASS_NAMES), len(CLASS_NAMES)).float()
confusion_fraction = confusion / confusion.sum(dim=1, keepdim=True).clamp_min(1)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
image = ax.imshow(confusion_fraction.cpu(), vmin=0, vmax=1, cmap="Blues")
for row in range(len(CLASS_NAMES)):
    for column in range(len(CLASS_NAMES)):
        value = float(confusion_fraction[row, column].cpu())
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", color="white" if value > 0.55 else "black")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set(xlabel="predicted class", ylabel="true class", title=f"Held-out probe confusion at t={REFERENCE_T}")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

reference_direction = directions[REFERENCE_T]
cosine_table = pd.DataFrame([
    {"t": t_value, f"cosine with t={REFERENCE_T:.2f} direction": float(torch.dot(direction, reference_direction).cpu())}
    for t_value, direction in directions.items()
])
display(cosine_table.round(3))

cosine_column = f"cosine with t={REFERENCE_T:.2f} direction"
fig, ax = plt.subplots(figsize=(7.2, 3.9))
ax.plot(cosine_table["t"], cosine_table[cosine_column], marker="o", lw=2, color="#0072B2")
ax.axhline(0, color="#777777", lw=1, ls="--")
ax.axvspan(0.40, 0.90, color="#E69F00", alpha=0.16, label="activation-only window")
ax.scatter([REFERENCE_T], [1.0], s=90, color="#D55E00", zorder=3, label="$d_{0.90}$ reference")
ax.set(
    xlabel="t (noise to data)", ylabel="cosine similarity",
    title="A late activation direction is not the local direction at every time",
    xlim=(0.05, 0.95), ylim=(-0.35, 1.08),
)
ax.grid(alpha=0.2)
ax.legend(frameon=False, loc="lower right")
plt.show()


### Reading fixed-direction transfer

We estimate $d_{0.90}$ at late time and reuse it as a fixed intervention vector
over the window $[0.40, 0.90]$. The cosine plot shows that time-specific
directions are only weakly aligned with $d_{0.90}$ near the beginning of that
window. The intervention therefore does not follow the local class direction
at every time. It asks a narrower question: can one late-time direction,
repeated over part of the trajectory, produce a useful endpoint change?


In [ ]:
def direction_statistics(direction):
    scores = test_features @ direction
    positive = scores[test_labels == TARGET_CLASS]
    negative = scores[test_labels != TARGET_CLASS]
    gap = positive.mean() - negative.mean()
    pooled_std = torch.sqrt(0.5 * (positive.var(unbiased=False) + negative.var(unbiased=False))).clamp_min(1e-8)
    order = torch.argsort(scores)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, scores.numel() + 1, device=device, dtype=torch.float32)
    n_positive = positive.numel()
    n_negative = negative.numel()
    auc = (
        ranks[test_labels == TARGET_CLASS].sum() - n_positive * (n_positive + 1) / 2
    ) / (n_positive * n_negative)
    return float(gap.cpu()), float((gap / pooled_std).cpu()), float(auc.cpu())

shuffled_directions = []
null_direction_rows = []
for null_index in range(50):
    set_seed(SEED + 4000 + null_index)
    shuffled_train_labels = train_labels[torch.randperm(train_labels.numel(), device=device)]
    direction = discriminant_direction(train_features, shuffled_train_labels, TARGET_CLASS)
    shuffled_directions.append(direction)
    raw_gap, normalized_gap, auc = direction_statistics(direction)
    null_direction_rows.append({
        "shuffle": null_index,
        "held-out raw gap": raw_gap,
        "held-out normalized gap": normalized_gap,
        "held-out AUC": auc,
    })

shuffled_direction = shuffled_directions[0]
true_raw_gap, true_normalized_gap, true_auc = direction_statistics(reference_direction)
null_direction_table = pd.DataFrame(null_direction_rows)
direction_summary = pd.DataFrame({
    "direction": ["true labels", "50 shuffled-label mean", "largest shuffled-label gap"],
    "normalized held-out gap": [
        true_normalized_gap,
        null_direction_table["held-out normalized gap"].mean(),
        null_direction_table["held-out normalized gap"].max(),
    ],
    "held-out AUC": [
        true_auc,
        null_direction_table["held-out AUC"].mean(),
        null_direction_table.loc[null_direction_table["held-out normalized gap"].idxmax(), "held-out AUC"],
    ],
})
display(direction_summary.round(3))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(null_direction_table["held-out normalized gap"], bins=12, color="#999999", alpha=0.75, label="50 shuffled-label directions")
ax.axvline(true_normalized_gap, color="#D55E00", lw=2.5, label="true-label direction")
ax.set(xlabel="normalized target-vs-rest gap on held-out activations", ylabel="count", title="Direction null distribution")
ax.legend(frameon=False)
plt.show()


## 3. Activation intervention

The target direction is covariance-aware: it separates target examples from
the remaining classes in hidden space. During an active time window, the model
adds a scaled unit direction to the selected hidden layer. The scale is relative
to the current feature RMS, so it remains comparable across time.

This is a genuine hidden-feature intervention, not sample-space attraction. It
is still simpler than NA-RFM: the paper uses RFM directions in image-model
activation tensors, while this notebook uses a linear discriminant direction in
one MLP layer.

We collect the direction at `t = 0.90`, where the held-out probe is strongest,
then reuse it over a wider intervention window. Collection time and
intervention time are different choices: the cosine plot above shows that the
late direction is not the local class direction at every step. The paired
comparison below asks whether this fixed edit is nevertheless useful.


In [ ]:
set_seed(SEED + 50)
x0 = sample_base(1400, device=device)
target_reference = sample_class(1400, TARGET_CLASS, device=device)

ACTIVATION_STRENGTH = 7.0
ACTIVATION_WINDOW = (0.40, 0.90)
COMBINED_ACTIVATION_STRENGTH = 5.0
COMBINED_ACTIVATION_WINDOW = (0.35, 0.85)

methods = {
    "baseline": base_velocity(model),
    "activation 0.0": make_activation_steered_velocity(
        model, reference_direction, strength=0.0,
        start=ACTIVATION_WINDOW[0], end=ACTIVATION_WINDOW[1],
    ),
    "shuffled activation": make_activation_steered_velocity(
        model, shuffled_direction, strength=ACTIVATION_STRENGTH,
        start=ACTIVATION_WINDOW[0], end=ACTIVATION_WINDOW[1],
    ),
    "activation steering": make_activation_steered_velocity(
        model, reference_direction, strength=ACTIVATION_STRENGTH,
        start=ACTIVATION_WINDOW[0], end=ACTIVATION_WINDOW[1],
    ),
    "noise alignment": make_noise_aligned_velocity(
        model, stats, target_class=TARGET_CLASS, strength=1.6
    ),
    "gradient guidance": make_gradient_guided_velocity(
        model, stats, target_class=TARGET_CLASS, strength=3.0, start=0.15, end=0.92
    ),
    "noise + activation": make_combined_velocity(
        model, stats, reference_direction,
        target_class=TARGET_CLASS, noise_strength=1.6,
        activation_strength=COMBINED_ACTIVATION_STRENGTH,
        activation_window=COMBINED_ACTIVATION_WINDOW,
    ),
}

rows = []
paths = {}
for name, velocity in methods.items():
    path, seconds = timed_sample(velocity, x0, steps=56, method="heun")
    metrics = evaluate_samples(path[-1].to(device), target_reference, stats, target_class=TARGET_CLASS)
    rows.append({"method": name, **metrics, "seconds": seconds})
    paths[name] = path

comparison_table = pd.DataFrame(rows)
comparison_table["runtime / baseline"] = comparison_table["seconds"] / comparison_table.loc[0, "seconds"]
zero_activation_error = float(torch.max(torch.abs(paths["baseline"][-1] - paths["activation 0.0"][-1])))
assert zero_activation_error < 1e-6
assert model_state_digest(model) == checkpoint_digest_before

null_control_table = comparison_table[
    comparison_table["method"].isin(["baseline", "activation 0.0", "shuffled activation", "activation steering"])
]
method_table = comparison_table[
    comparison_table["method"].isin(["baseline", "noise alignment", "gradient guidance", "activation steering", "noise + activation"])
]
display(null_control_table.round(4))
display(method_table.round(4))


In [ ]:
plot_trajectory_comparison(
    {
        "baseline": paths["baseline"],
        "shuffled direction": paths["shuffled activation"],
        "activation steering": paths["activation steering"],
        "noise + activation": paths["noise + activation"],
    },
    target_reference=target_reference[:700],
    n_lines=18,
)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 5))
marker_by_method = {
    "baseline": "o", "shuffled activation": "s",
    "activation steering": "D", "noise alignment": "^", "gradient guidance": "v",
    "noise + activation": "P",
}
label_offsets = {
    "baseline": (-52, -12),
    "shuffled activation": (5, 8),
    "activation steering": (5, 4),
    "noise alignment": (5, 4),
    "gradient guidance": (5, 4),
    "noise + activation": (5, -2),
}
tradeoff_table = comparison_table[comparison_table["method"] != "activation 0.0"]
for _, row in tradeoff_table.iterrows():
    ax.scatter(row["target_wasserstein"], row["target_rate"], s=80, marker=marker_by_method[row["method"]])
    ax.annotate(
        row["method"], (row["target_wasserstein"], row["target_rate"]),
        xytext=label_offsets[row["method"]], textcoords="offset points", fontsize=9,
    )
ax.set(xlabel="Wasserstein distance to target class (lower is better)", ylabel="target-class rate (higher is better)", title="Control-quality trade-off")
ax.grid(alpha=0.2)
plt.show()


In [ ]:
method_rows = comparison_table.set_index("method")
activation_gain = method_rows.loc["activation steering", "target_rate"] - method_rows.loc["baseline", "target_rate"]
shuffled_gain = method_rows.loc["shuffled activation", "target_rate"] - method_rows.loc["baseline", "target_rate"]
activation_wasserstein_reduction = method_rows.loc["baseline", "target_wasserstein"] - method_rows.loc["activation steering", "target_wasserstein"]

activation_effects = pd.DataFrame({
    "comparison": [
        "activation steering minus baseline",
        "shuffled activation minus baseline",
        "activation steering: Wasserstein reduction",
    ],
    "observed change": [
        activation_gain,
        shuffled_gain,
        activation_wasserstein_reduction,
    ],
})
display(activation_effects.set_index("comparison").round(4))


## 4. Reading the method comparison

The methods intervene at different places. Noise alignment changes a denoised
estimate early, objective-gradient guidance differentiates a class score, and
activation steering edits a hidden feature. Target-class rate describes
control, Wasserstein distance describes agreement with target examples,
diversity describes spread, and runtime exposes the price of online
backpropagation. No method is best without naming the trade-off that matters.

**Questions for discussion**

1. Does probe accuracy predict steering success at the same collection time?
2. Over which times is the learned direction stable enough to reuse?
3. When does combining coarse high-noise and later activation control help?
4. Which conclusion from 2D would require a new experiment before claiming it for images?

The probe and intervention answer different questions: held-out decoding shows
that class information is readable from the hidden layer, while paired
sampling tests whether one fixed late-time direction is useful over the chosen
intervention window. Neither result implies a unique class axis or a
time-invariant representation. Across the toy series, we have now compared
three distinct intervention sites. Notebook `06` carries only noise alignment
to a pretrained image model and asks whether paired images and batch-level
features still reveal a measurable, partial shift.
